# 09 — Refresh dashboard observations offline

After completed 08 with your longer-history checkpoint, select the project kernel and **Restart Kernel and Run All**. This is maintenance, not a new model experiment. It adds recent observations, applies frozen rules/models and precomputes explanations. No fitting, threshold selection, label creation or model evaluation takes place.

The default selects the last calendar week fully closed before today in Malaysia. You may set `SNAPSHOT_WEEK` to an earlier Monday for a dated demonstration. A snapshot is scored **now**, not backdated: it is not evidence that a forecast was issued before its outcomes began. Source access is required for a new snapshot; a complete verified cache is reused.

The original 08 export, source/prepared files, models and research registration are preserved. A separate `08_dashboard/refresh_manifest.json` is published **last**, after checks pass. The app uses it when present. Failed downloads do not replace a working export. A registered outcome month is never opened by this workflow; handle the reserved comparison separately.


**Execution status:** saved outputs describe the last local run. Rerun this notebook for an executed copy of the current checks and coverage comparison.

In [1]:
import ast
import hashlib
import io
import json
from datetime import datetime
from importlib import metadata, reload
from pathlib import Path
from urllib.request import urlopen
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd
from scipy.special import expit
import pricecatcher_config as cfg
cfg = reload(cfg)

SNAPSHOT_WEEK = None  # None = latest fully closed week; or an earlier Monday, e.g. "2026-10-12".
now = pd.Timestamp.now(tz="Asia/Kuala_Lumpur")
today = now.tz_localize(None).normalize()
week_end = today - pd.Timedelta(days=today.dayofweek + 1)
week_start = week_end - pd.Timedelta(days=6)
if SNAPSHOT_WEEK is not None:
    week_start = pd.Timestamp(SNAPSHOT_WEEK).normalize()
    week_end = week_start + pd.Timedelta(days=6)
if week_start.tzinfo is not None or week_start.dayofweek != 0 or week_end >= today:
    raise ValueError("Choose an earlier Monday whose entire week is closed before today in Malaysia.")


def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def verify(path, expected):
    if not path.is_file() or file_hash(path) != expected:
        raise ValueError(f"Frozen file missing/changed: {path}. Restore it; nothing is refitted here.")


def notebook_functions(filename, names):
    """Reuse named calculations only; never run other notebook cells."""
    notebook = json.loads((Path(cfg.__file__).parent / filename).read_text())
    definitions = []
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            definitions += [node for node in ast.parse("".join(cell["source"])).body
                            if isinstance(node, ast.FunctionDef) and node.name in names]
    if {node.name for node in definitions} != set(names):
        raise ValueError(f"Expected calculation definitions are missing from {filename}.")
    exec(compile(ast.Module(body=definitions, type_ignores=[]), filename, "exec"), globals())


notebook_functions("05_features_and_labels.ipynb", ["make_features"])
notebook_functions("08_shap_and_dashboard_inputs.ipynb", ["model_inputs", "dense_matrix", "source_features", "explain_rows", "anomaly_inputs"])
print("Run:", cfg.PROJECT_ROOT)
print("Observed week:", week_start.date(), "to", week_end.date(), "| Run started at:", now.isoformat())


Run: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Observed week: 2026-10-12 to 2026-10-18 | Run started at: 2026-10-21T02:06:33.272238+08:00


## 1. Verify frozen inputs and protect the registered period

The original dashboard export selects the checkpoint. Saved preprocessing, item eligibility, bounds, unit conversions and score cutoff are retained. Unknown historical shop metadata is audited and excluded; it is never guessed. The frozen original lookup is a limitation for newly added shops.


In [2]:
dashboard_dir = cfg.PROJECT_ROOT / "08_dashboard"
original_entry = dashboard_dir / "dashboard_manifest.json"
base = json.loads(original_entry.read_text())
model_record = base["forecast_model"]
if model_record["version"] != "longer_2024_2025":
    raise ValueError("This refresh supports your completed longer-history prototype. Finish research 12 and export it in 08 first.")
base_cache = dashboard_dir / base["cache_dir"]
base_manifest_path = base_cache / "explanation_manifest.json"
verify(base_manifest_path, base["cache_manifest_sha256"])
base_manifest = json.loads(base_manifest_path.read_text())
if base_manifest["forecast_model"] != model_record:
    raise ValueError("Original dashboard cache and model metadata differ.")
for name, expected in base_manifest["output_sha256"].items():
    verify(base_cache / name, expected)
model_path = (cfg.PROJECT_ROOT / model_record["model_path"]).resolve()
if not model_path.is_relative_to(cfg.PROJECT_ROOT.resolve()):
    raise ValueError("Checkpoint must belong to this run.")
verify(model_path, model_record["sha256"])
experiment_path = model_path.parent / "experiment_manifest.json"
experiment = json.loads(experiment_path.read_text())
if experiment.get("prepared_test_accessed") is not False or experiment["protocol"]["cutoff"] != model_record["score_cutoff"]:
    raise ValueError("Longer-history checkpoint/cutoff differs from the completed experiment.")
verify(model_path, experiment["output_sha256"][model_path.name])
for package, expected in experiment["protocol"]["package_versions"].items():
    if metadata.version(package) != expected:
        raise ValueError(f"Restore the checkpoint's {package} version before inference.")
prepared = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
cleaning_path = prepared.parent / "03_cleaned" / "cleaning_summary.json"
daily_path = prepared.parent / "03_cleaned" / "daily_prices.parquet"
for path in [prepared / "experiment_settings.json", prepared / "train.parquet", cleaning_path, daily_path]:
    verify(path, experiment["protocol"]["input_sha256"][str(path.resolve())])
settings = json.loads((prepared / "experiment_settings.json").read_text())
cleaning = json.loads(cleaning_path.read_text())
if cleaning["preparation"] != settings["preparation"] or settings["remove_flagged_prices"]:
    raise ValueError("Expected the completed retained-flags cleaning policy; no new cleaning policy is learned here.")
start = pd.Timestamp(settings["evaluation_end"]) + pd.Timedelta(days=1)
if week_end < start:
    raise ValueError("Choose a snapshot after the original source scope; use 08 to reopen the original snapshot.")
months = pd.period_range(start, week_end, freq="M").astype(str).tolist()
registration_paths = sorted((cfg.PROJECT_ROOT / "13_registered_future_evaluation").glob("*/registration.json"))
for path in registration_paths:
    registration = json.loads(path.read_text())
    content = {k: v for k, v in registration.items() if k != "registration_sha256"}
    if hashlib.sha256(json.dumps(content, sort_keys=True).encode()).hexdigest() != registration["registration_sha256"]:
        raise ValueError("Registered protocol changed. Restore it; do not bypass the reserved-period guard.")
    design = registration["design"]
    reserved = pd.period_range(design["outcome_start"], design["outcome_end"], freq="M").astype(str)
    if set(months) & set(reserved):
        raise ValueError("Refresh would open a registered outcome month. Keep the last published snapshot; handle the reserved comparison separately. No source was downloaded.")
source = cfg.verify_source_snapshot()
if source["source_snapshot_sha256"] != settings["preparation"]["source_snapshot_sha256"]:
    raise ValueError("Frozen source snapshot differs from the completed preparation.")
anomaly_dir = cfg.PROJECT_ROOT / "07_anomalies"
anomaly_manifest_path = anomaly_dir / "anomaly_manifest.json"
anomaly_record = json.loads(anomaly_manifest_path.read_text())
anomaly_path = anomaly_dir / "isolation_forest.joblib"
verify(anomaly_path, anomaly_record["model_sha256"])
if anomaly_record["model_sha256"] != base["anomaly_model"]["sha256"] or anomaly_record["protocol"]["schema_version"] != 1:
    raise ValueError("Anomaly detector does not match the original dashboard.")
for package, expected in anomaly_record["package_versions"].items():
    if metadata.version(package) != expected:
        raise ValueError(f"Restore the anomaly detector's {package} version.")
protected = [original_entry, base_manifest_path, model_path, experiment_path, anomaly_path, anomaly_manifest_path,
             prepared / "experiment_settings.json", prepared / "train.parquet", cleaning_path, daily_path]
protected += [base_cache / name for name in base_manifest["output_sha256"]]
protected += registration_paths
protected += [cfg.PROJECT_ROOT / "01_integrated" / "source_manifest.json"]
protected += list((cfg.PROJECT_ROOT / "01_integrated" / "lookups").glob("*.parquet"))
protected_hashes = {str(p.relative_to(cfg.PROJECT_ROOT)): file_hash(p) for p in protected}
print("Frozen checkpoints and registration verified. Source months:", months)


Frozen checkpoints and registration verified. Source months: ['2026-07', '2026-08', '2026-09', '2026-10']


## 2. Download only the selected basket and audit attrition

Each complete snapshot source cache is immutable and hash checked. Monthly payloads are processed one at a time and raw downloads are not saved. The current-month payload can contain dates later than the snapshot; these are counted and excluded before aggregation. This is a retrospective demonstration, not an untouched evaluation period.

Invalid prices, missing required metadata, frozen wholesale exclusions and frozen low-coverage items are counted separately. Statistical flags use the original training bounds and remain in the model inputs. They do not confirm errors. No reviewed source price is silently replaced.


In [3]:
lookup_dir = cfg.PROJECT_ROOT / "01_integrated" / "lookups"
item = pd.read_parquet(lookup_dir / "lookup_item.parquet")
premise = pd.read_parquet(lookup_dir / "lookup_premise.parquet")
for table, key in [(item, "item_code"), (premise, "premise_code")]:
    if table[key].isna().any() or table[key].duplicated().any():
        raise ValueError("Frozen lookup keys must be complete and unique.")
item_map = {int(k): v for k, v in settings["preparation"]["item_map"].items()}
egg_divisors = {118: 1, 119: 1, 120: 1, 1109: 3, 1110: 3, 1111: 3}
units = item.set_index("item_code").unit.astype("string").str.strip().str.lower()
for code, divisor in egg_divisors.items():
    if units.get(code) != ("30 biji" if divisor == 3 else "10 biji"):
        raise ValueError("Frozen egg lookup disagrees with the conversion contract.")
item = item[["item_code", "item", "unit", "item_group", "item_category"]]
premise = premise[["premise_code", "premise", "state", "district", "premise_type"]]
bounds = pd.DataFrame(cleaning["training_price_bounds"])[["item_code", "lower_bound", "upper_bound"]]
source_policy = {"start": str(start.date()), "end": str(week_end.date()), "months": months,
    "frozen_snapshot": source["source_snapshot_sha256"], "cleaning_sha256": file_hash(cleaning_path),
    "unknown_metadata": "exclude with audit; frozen lookup only", "statistical_flags": "retain using frozen bounds"}
source_id = hashlib.sha256(json.dumps(source_policy, sort_keys=True).encode()).hexdigest()[:16]
source_dir = cfg.PROJECT_ROOT / "09_refresh" / str(week_start.date()) / source_id
source_dir.mkdir(parents=True, exist_ok=True)
source_path = source_dir / "selected_daily_prices.parquet"
source_record_path = source_dir / "source_manifest.json"
if source_record_path.exists():
    source_record = json.loads(source_record_path.read_text())
    if source_record["policy"] != source_policy:
        raise ValueError("Refresh source policy changed. Preserve the existing snapshot.")
    verify(source_path, source_record["selected_sha256"])
    print("Reusing verified refresh source; no download.")
else:
    frames, audit = [], []
    for month in months:
        url = cfg.BASE_PRICE_URL.format(month=month)
        print("Downloading:", month)
        with urlopen(url, timeout=60) as response:
            if response.status != 200:
                raise ValueError("A monthly source download did not complete.")
            payload = response.read(64 * 1024 * 1024 + 1)
            etag = response.headers.get("ETag")
            length = response.headers.get("Content-Length")
        if len(payload) > 64 * 1024 * 1024 or (length is not None and len(payload) != int(length)):
            raise ValueError("Monthly source exceeded the size limit or was incomplete; no export is published.")
        prices = pd.read_parquet(io.BytesIO(payload), columns=["date", "premise_code", "item_code", "price"])
        raw_rows = len(prices)
        prices["date"] = pd.to_datetime(prices.date, errors="raise").dt.normalize()
        if prices.date.isna().any() or not prices.date.dt.to_period("M").eq(pd.Period(month)).all():
            raise ValueError("Monthly source has missing dates or records outside its declared month.")
        later_rows = int(prices.date.gt(week_end).sum())
        prices = prices.loc[prices.date.between(start, week_end)].copy()
        prices["item_code"] = pd.to_numeric(prices.item_code, errors="raise").astype("Int64")
        prices = prices.loc[prices.item_code.isin(item_map)].copy()
        prices["premise_code"] = pd.to_numeric(prices.premise_code, errors="raise").astype("Int64")
        if prices.duplicated(["date", "premise_code", "item_code"]).any():
            raise ValueError("Duplicate daily source keys need review; no automatic deduplication.")
        prices["price"] = pd.to_numeric(prices.price, errors="coerce").astype(float)
        prices["project_group"] = prices.item_code.map(item_map)
        data = prices.merge(item, on="item_code", how="left", validate="many_to_one")
        data = data.merge(premise, on="premise_code", how="left", validate="many_to_one")
        for column in cfg.TEXT_COLS:
            data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
        for code, divisor in egg_divisors.items():
            data.loc[data.item_code.eq(code), "price"] /= divisor
            data.loc[data.item_code.eq(code), "unit"] = "10 biji"
        invalid = ~np.isfinite(data.price) | data.price.le(0)
        unknown = data[cfg.TEXT_COLS + ["premise_code"]].isna().any(axis=1) & ~invalid
        wholesale = (data.premise_type.eq("Borong").fillna(False) & ~invalid & ~unknown
                     if settings["preparation"]["remove_borong"] else pd.Series(False, index=data.index))
        sparse = data.item_code.isin(settings["excluded_item_codes"]) & ~invalid & ~unknown & ~wholesale
        retained = data.loc[~invalid & ~unknown & ~wholesale & ~sparse].merge(bounds, on="item_code", how="left", validate="many_to_one")
        if retained[["lower_bound", "upper_bound"]].isna().any().any():
            raise ValueError("An eligible item has no frozen training bounds.")
        retained["is_suspicious_price"] = retained.price.lt(retained.lower_bound) | retained.price.gt(retained.upper_bound)
        audit.append({"month": month, "url": url, "raw_sha256": hashlib.sha256(payload).hexdigest(), "etag": etag,
            "raw_rows": raw_rows, "rows_after_snapshot_excluded": later_rows, "selected_scope_rows": len(data),
            "invalid_removed": int(invalid.sum()), "unknown_metadata_removed": int(unknown.sum()),
            "unknown_premise_sample": data.loc[unknown, "premise_code"].dropna().drop_duplicates().head(10).tolist(),
            "wholesale_removed": int(wholesale.sum()), "frozen_sparse_items_removed": int(sparse.sum()),
            "retained_rows": len(retained), "flagged_rows_retained": int(retained.is_suspicious_price.sum())})
        if len(data) != int(invalid.sum() + unknown.sum() + wholesale.sum() + sparse.sum()) + len(retained):
            raise ValueError("Source attrition counts do not reconcile.")
        frames.append(retained[cfg.FINAL_COLS + ["is_suspicious_price"]])
        del prices, data, retained, payload
    selected = pd.concat(frames, ignore_index=True)
    if selected.empty or selected.date.max() < week_end:
        raise ValueError("The source has not supplied observations through the snapshot end. Keep the previous export or explicitly select an earlier Monday.")
    temporary = source_path.with_suffix(".tmp")
    selected.to_parquet(temporary, index=False)
    temporary.replace(source_path)
    source_record = {"policy": source_policy, "monthly_audit": audit, "selected_rows": len(selected),
        "selected_sha256": file_hash(source_path), "source_correctness": "unverified; frozen flags retained, unknown metadata quarantined"}
    temporary = source_record_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(source_record, indent=2))
    temporary.replace(source_record_path)
    del frames, selected
source_hash = file_hash(source_record_path)
display(pd.DataFrame(source_record["monthly_audit"]))


Reusing verified refresh source; no download.


,month,url,raw_sha256,etag,raw_rows,rows_after_snapshot_excluded,selected_scope_rows,invalid_removed,unknown_metadata_removed,unknown_premise_sample,wholesale_removed,frozen_sparse_items_removed,retained_rows,flagged_rows_retained
0,2026-07,https://storage.data.gov.my/pricecatcher/price...,750f2a668937606f0d8c9fb07b58f29f6f04ed10c7bb8b...,"""aabcf008af2a3768c6a40069fce7700a""",1815849,0,302199,0,0,[],0,548,301651,60
1,2026-08,https://storage.data.gov.my/pricecatcher/price...,d3f6bcf3b6ffc1b32c86ded4275516b9fa5961d4113235...,"""5516b505e06b9e672a6dc255f3acadd1""",1933285,0,298769,0,0,[],0,542,298227,59
2,2026-09,https://storage.data.gov.my/pricecatcher/price...,30cad2b7cd5199f654df5b588155f16aacd6f29fb89002...,"""b14b0365260c1438ef2387974d2bab11""",1742320,0,279595,0,4,[22280],0,370,279221,73
3,2026-10,https://storage.data.gov.my/pricecatcher/price...,3a9a92a02744c8d328a36c78be213913138c0fd214a250...,"""a81e68a0a7131fd859e5d9f4adcf885c""",746995,236353,86395,0,0,[],0,95,86300,13


## 3. Build exact-week features and check the original reconstruction

Reuse the feature calculation from 05 and the scoring/explanation calculations from 08. The original June inputs must reconstruct before new scores are accepted. Outcome-audit columns returned by 05 are discarded; no surge labels or evaluation scores are created. New observations only use dates up to the selected origin-week end.


The reconstruction check allows `1e-6` RM absolute numerical tolerance only for the two rolling standard deviations. A shorter historical prefix can leave tiny differences near zero in pandas rolling variance. Other columns retain the strict `1e-9` tolerance; missing-value patterns and row keys must still match. No saved feature or model input is rounded or replaced.

The support tables compare this snapshot with the original June snapshot by food group and state. Changes can reflect monitoring availability or eligibility; they do not measure accuracy or population coverage. Investigate reduced support without relaxing the frozen rules just to increase row counts.

In [4]:
context_start = start - pd.Timedelta(weeks=10)
context_start -= pd.Timedelta(days=context_start.dayofweek)
columns = cfg.GROUP_KEYS + ["date", "premise_code", "price", "is_suspicious_price"]
context = pd.read_parquet(daily_path, columns=columns, filters=[("date", ">=", context_start)])
recent = pd.read_parquet(source_path, columns=columns)
daily = pd.concat([context, recent], ignore_index=True)
daily["date"] = pd.to_datetime(daily.date)
daily = daily.loc[daily.date.le(week_end)].copy()
if daily.duplicated(["date", "premise_code", "item_code"]).any():
    raise ValueError("Context and new observations have overlapping source keys.")
daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
keys = cfg.GROUP_KEYS + ["week_start"]
weekly = daily.groupby(keys).agg(weekly_median_price=("price", "median"), weekly_std_price=("price", "std"),
    record_count=("price", "size"), premise_count=("premise_code", "nunique"), observed_days=("date", "nunique"),
    suspicious_record_count=("is_suspicious_price", "sum")).reset_index()
weekly = weekly.loc[weekly.week_start.ge(context_start) & (weekly.week_start + pd.Timedelta(days=6)).le(week_end)]
parts = [make_features(part) for _, part in weekly.groupby(cfg.GROUP_KEYS, sort=False)]
featured = pd.concat(parts, ignore_index=True)
featured = featured.drop(columns=featured.filter(regex="^next_week_").columns)
featured["month"] = featured.week_start.dt.month
week_number = featured.week_start.dt.isocalendar().week.astype(float)
featured["week_sin"] = np.sin(2 * np.pi * week_number / 52.1775)
featured["week_cos"] = np.cos(2 * np.pi * week_number / 52.1775)
supported = (featured.record_count.ge(settings["minimum_records"]) & featured.premise_count.ge(settings["minimum_premises"])
    & featured.lag_1w_price.notna() & featured.history_count_4w.ge(settings["minimum_history_4w"]))
eligible = featured.loc[supported].copy()
features = settings["feature_columns"]
original_latest = pd.read_parquet(base_cache / "latest_dashboard.parquet")
rebuilt = eligible.loc[eligible.week_start.eq(pd.Timestamp(base["latest_complete_week"]))]
check_columns = list(dict.fromkeys(keys + features))
expected = original_latest[check_columns].sort_values(keys).reset_index(drop=True)
actual = rebuilt[check_columns].sort_values(keys).reset_index(drop=True)
# Pandas rolling variance accumulates tiny rounding differences when the earlier
# history is shorter. Near zero, std() magnifies these through the square root.
# Use one millionth of RM as absolute tolerance for these columns only; never round inputs.
std_columns = ["rolling_std_4w", "rolling_std_8w"]
pd.testing.assert_frame_equal(expected.drop(columns=std_columns), actual.drop(columns=std_columns),
                              check_dtype=False, rtol=1e-9, atol=1e-9)
pd.testing.assert_frame_equal(expected[std_columns], actual[std_columns],
                              check_dtype=False, rtol=1e-9, atol=1e-6)
print("Maximum rolling-std reconstruction differences (RM per listed unit):")
display((expected[std_columns] - actual[std_columns]).abs().max().rename("max_absolute_difference"))
identifiers = cfg.GROUP_KEYS + ["week_start", "next_week_end"]
latest = eligible.loc[eligible.week_start.eq(week_start)].copy()
latest["next_week_end"] = week_start + pd.Timedelta(days=13)
if latest.empty:
    raise ValueError("No supported rows in this week. Keep the previous published snapshot; inspect coverage before selecting an earlier week.")
if not np.isfinite(latest[settings["numeric_features"]].fillna(0).to_numpy(dtype=float)).all():
    raise ValueError("Refresh contains non-finite model features.")
if latest.duplicated(keys).any():
    raise ValueError("Latest series keys must be unique.")
forecast_model = joblib.load(model_path)
detector = joblib.load(anomaly_path)
scores = forecast_model.predict_proba(model_inputs(latest, settings))[:, 1]
if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
    raise ValueError("The frozen model returned invalid scores; the published dashboard was not replaced.")
scored_at = pd.Timestamp.now(tz="Asia/Kuala_Lumpur").isoformat()
latest["model_score"] = scores
latest["predicted_surge"] = (scores >= model_record["score_cutoff"]).astype(int)
latest["forecast_model_version"] = model_record["version"]
X_anomaly = anomaly_inputs(latest)
if list(X_anomaly.columns) != list(anomaly_record["protocol"]["feature_formulas"]):
    raise ValueError("Anomaly feature definitions differ from the frozen detector.")
latest["anomaly_score"] = -detector.score_samples(X_anomaly)
if not np.isfinite(latest.anomaly_score.to_numpy(dtype=float)).all():
    raise ValueError("The frozen anomaly detector returned invalid scores; no refresh was published.")
latest["is_anomaly"] = detector.predict(X_anomaly) == -1
latest = latest.sort_values(["model_score"] + keys, ascending=[False] + [True] * len(keys)).reset_index(drop=True)
latest["forecast_id"] = [f"refresh_{week_start.date()}_{i:05d}" for i in range(len(latest))]
latest["has_cached_explanation"] = latest.index < 200
# Only observed-price/coverage columns are added to dashboard history.
history_columns = cfg.GROUP_KEYS + ["week_start", "weekly_median_price", "record_count", "premise_count", "observed_days"]
history = eligible.loc[eligible.week_start.gt(pd.Timestamp(base["latest_complete_week"])), history_columns].copy()
print("Original feature reconstruction passed. Supported refreshed rows:", len(latest))
print("No prices were corrected and no model/cutoff was changed.")

print("Supported rows in original snapshot:", len(original_latest), "| Refreshed snapshot:", len(latest))
for column in ["project_group", "state"]:
    coverage = pd.concat([original_latest.groupby(column).size().rename("original_supported_rows"),
                          latest.groupby(column).size().rename("refreshed_supported_rows")], axis=1).fillna(0).astype(int)
    coverage["change_in_supported_rows"] = coverage.refreshed_supported_rows - coverage.original_supported_rows
    display(coverage)
print("Support changes describe observed eligible series, not a change in prediction accuracy.")


Maximum rolling-std reconstruction differences (RM per listed unit):


rolling_std_4w    1.366187e-07
rolling_std_8w    2.594926e-07
Name: max_absolute_difference, dtype: float64

Original feature reconstruction passed. Supported refreshed rows: 2946
No prices were corrected and no model/cutoff was changed.


## 4. Explain and publish a separate derived snapshot

Only the highest-scoring 200 supported latest rows receive local explanations. The original common 2025 reference sample and its importance summary are preserved. Grouped contributions must reconstruct scores before the refreshed entry point replaces an earlier refresh pointer. The original `dashboard_manifest.json` is never overwritten.


In [5]:
training = pd.read_parquet(prepared / "train.parquet", columns=list(dict.fromkeys(cfg.GROUP_KEYS + features)))
background = training.sample(min(100, len(training)), random_state=42)
family = base_manifest["protocol"]["family"]
local = latest.head(200)
values, baseline, raw, explained_scores, _ = explain_rows(forecast_model, family, local, background, settings)
local_explanations = local[identifiers].copy()
local_explanations["cohort"] = "latest"
local_explanations["explanation_id"] = local.forecast_id.to_numpy()
local_explanations["base_value"], local_explanations["raw_output"], local_explanations["model_score"] = baseline, raw, explained_scores
local_explanations[["shap__" + name for name in features]] = values
old_explanations = pd.read_parquet(base_cache / "shap_explanations.parquet")
explanations = pd.concat([old_explanations.loc[old_explanations.cohort.eq("historical_reference")], local_explanations], ignore_index=True)
explanation_columns = ["shap__" + name for name in features] + ["base_value", "raw_output", "model_score"]
if not np.isfinite(explanations[explanation_columns].to_numpy(dtype=float)).all():
    raise ValueError("An explanation contains missing or infinite values; no refresh was published.")
np.testing.assert_allclose(explanations.base_value + explanations[["shap__" + name for name in features]].sum(axis=1),
                           explanations.raw_output, rtol=1e-5, atol=1e-5)
np.testing.assert_allclose(local.model_score, explained_scores, rtol=1e-6, atol=1e-8)
notebook_path = Path(cfg.__file__).parent / "09_refresh_dashboard.ipynb"
code_text = []
for path in [notebook_path, Path(cfg.__file__).parent / "05_features_and_labels.ipynb",
             Path(cfg.__file__).parent / "08_shap_and_dashboard_inputs.ipynb"]:
    notebook = json.loads(path.read_text())
    code_text.extend("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code")
code_hash = hashlib.sha256("\n".join(code_text).encode()).hexdigest()
shap_version = metadata.version("shap")
cache_id = hashlib.sha256(json.dumps({"source_sha256": source_hash, "model_sha256": model_record["sha256"],
    "original_manifest_sha256": file_hash(original_entry), "code_sha256": code_hash, "shap_version": shap_version}, sort_keys=True).encode()).hexdigest()[:16]
cache = dashboard_dir / f"refresh_{week_start.date()}_{cache_id}"
cache.mkdir(parents=True, exist_ok=True)
if (cache / "explanation_manifest.json").exists():
    manifest = json.loads((cache / "explanation_manifest.json").read_text())
    if manifest["protocol"]["refresh_source_sha256"] != source_hash or manifest["forecast_model"] != model_record:
        raise ValueError("Existing refresh cache describes different inputs.")
    for name, expected in manifest["output_sha256"].items():
        verify(cache / name, expected)
    refresh = manifest["refresh"]
    files = list(manifest["output_sha256"])
    print("Reusing the completed derived cache and its original scoring timestamp.")
else:
    latest.to_parquet(cache / "latest_dashboard.parquet", index=False)
    explanations.to_parquet(cache / "shap_explanations.parquet", index=False)
    (cache / "reference_feature_importance.csv").write_bytes((base_cache / "reference_feature_importance.csv").read_bytes())
    history.to_parquet(cache / "refresh_history.parquet", index=False)
    files = ["latest_dashboard.parquet", "shap_explanations.parquet", "reference_feature_importance.csv", "refresh_history.parquet"]
    refresh = {"snapshot_type": "retrospective offline refresh; not a timely or independently evaluated forecast",
        "generated_at": scored_at, "origin_week": str(week_start.date()),
        "original_entry_sha256": file_hash(original_entry), "source_manifest": str(source_record_path.relative_to(cfg.PROJECT_ROOT)),
        "source_manifest_sha256": source_hash, "frozen_lookup_used": True, "source_correctness": "unverified; see monthly audit",
        "history_scope": "original eligible history plus refreshed eligible observations; missing/excluded weeks remain gaps",
        "registered_period_loaded": False, "model_fitted": False, "cutoff_changed": False}
    protocol = {**base_manifest["protocol"], "shap_version": shap_version,
    "reconstruction_tolerance": {"rtol": 1e-9, "default_atol": 1e-9, "rolling_std_atol": 1e-6},
    "refresh_source_sha256": source_hash, "refresh_code_sha256": code_hash}
    manifest = {**{k: v for k, v in base_manifest.items() if k != "offline_cache_build_seconds"},
        "protocol": protocol, "refresh": refresh,
        "latest_complete_week": str(week_start.date()), "latest_supported_rows": len(latest), "latest_explained_rows": len(local),
        "upstream_sha256": {**base_manifest["upstream_sha256"], **protected_hashes,
                            str(source_record_path.relative_to(cfg.PROJECT_ROOT)): source_hash},
        "output_sha256": {name: file_hash(cache / name) for name in files}}
    (cache / "explanation_manifest.json").write_text(json.dumps(manifest, indent=2))
for relative, expected in protected_hashes.items():
    verify(cfg.PROJECT_ROOT / relative, expected)
verify(source_path, source_record["selected_sha256"])
verify(source_record_path, source_hash)
published = {**base, "cache_dir": cache.name, "cache_manifest_sha256": file_hash(cache / "explanation_manifest.json"),
    "latest_complete_week": str(week_start.date()), "latest_supported_rows": len(latest), "refresh": refresh,
    "files": {name: str(Path(cache.name) / name) for name in files}}
temporary = dashboard_dir / "refresh_manifest.tmp"
temporary.write_text(json.dumps(published, indent=2))
temporary.replace(dashboard_dir / "refresh_manifest.json")
print("Published refreshed dashboard snapshot:", week_start.date())
print("Restart/reload the dashboard. Original 08 export and registered comparison remain unchanged.")
print("Source audit:", source_record_path)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reusing the completed derived cache and its original scoring timestamp.
Published refreshed dashboard snapshot: 2026-10-12
Restart/reload the dashboard. Original 08 export and registered comparison remain unchanged.
Source audit: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/09_refresh/2026-10-12/c5b54c5de6259c34/source_manifest.json


## Read the refreshed dashboard correctly

Newer dates improve freshness, not proven forecasting accuracy. Scores were generated at the recorded time and are retrospective; the refresh does not archive timely registered forecasts or evaluate their outcomes. Review coverage and source attrition, especially unknown shop codes. Model-group limitations and uncalibrated scores remain.

Your 12 training-source cases in research 14 are still unresolved without independent evidence. This workflow does not invent replacement prices or treat statistical flags as confirmed errors. Further confirmed corrections belong in a separate documented experiment.

To reopen the original June demonstration intentionally, remove **only** `08_dashboard/refresh_manifest.json` (keep archived caches), then restart the app. Do not delete the original manifest, prepared data or models. Running 08 alone does not remove an existing refresh pointer.
